# Voice-of-Customer Agent - Build-Along

**Group 1 · Level: Intermediate · ~40 min**

**The pain:** Feedback about Optia products and clinics arrives from everywhere - app reviews, support tickets, ECP comments, survey free-text. Nobody can read it all, so patterns (and safety signals) get missed until they become problems.

**Your mission:** Build an agent that ingests a batch of raw feedback, groups it into themes, ranks them by urgency × frequency, and hands the product team a clear "here's what to pay attention to" summary - while never letting a safety comment slip through.

> **What "great" looks like:** The demo visibly catches a safety-related comment and flags it for human review - even when the model alone might have missed it. Defense in depth: the guardrail is a rule, not a hope.

## How this works - build it, then measure it

You won't build blind. You'll build an agent **and a way to score it**, then improve it in stages and watch the score climb.

**The agent** is a model that runs in a loop: it can call **tools** (so it can *act*, not just talk), and a **guardrail** wraps the loop as a safety net.

**The method** is *evals first*: define test tasks + graders, measure a **baseline**, then improve and **re-measure**. You'll make three small edits - the **brief**, the **tool descriptions**, and **one line of the guardrail** - and after each you re-run the eval and the bar fills:

| Stage | You add | Expect |
|---|---|---|
| 0 · Baseline | *(given)* a naive prompt, no tools | a low score - the starting line |
| 1 · Instructions | the brief (plain English) | ↑ |
| 2 · Tools | describe the agent's tools | ↑↑ |
| 3 · Guardrail | fill one safety-check line | ↑↑↑ target |

Run the cells top to bottom. Each cell prints what it did; green banners = your edit passed its check.

## Step 0 · Setup - connect to the model

Run this first. Paste your key into the `.env` when prompted (never into a cell). A green banner means you're connected.

In [ ]:
# ── Install packages + connect to the AI model ──
# Installs into THIS kernel; survives locked-down (PEP 668) Pythons. Safe to re-run.
import importlib.util, subprocess, sys

def _ensure_packages(reqs):
    missing = [pip for mod, pip in reqs if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " - first run only…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        if subprocess.run(base + extra + missing, capture_output=True, text=True).returncode == 0:
            return
    raise SystemExit("Could not install " + ", ".join(missing) + " - see SETUP.md (venv path fixes this).")

# No third-party packages needed - the model is called over plain HTTPS with the standard library.
import json, os, pathlib, time, urllib.request, urllib.error

# --- tiny zero-dependency model client (OpenAI + Anthropic) ---
class _Fn:
    def __init__(self, name, arguments): self.name = name; self.arguments = arguments
class _ToolCall:
    def __init__(self, id, name, arguments): self.id = id; self.function = _Fn(name, arguments)
class _Reply:
    def __init__(self, text, tool_calls): self.text = text; self.tool_calls = tool_calls

def _provider(model):
    m = model.lower()
    if m.startswith("anthropic/") or "claude" in m: return "anthropic"
    if m.startswith("openai/") or m.startswith(("gpt", "o1", "o3", "o4")): return "openai"
    return "anthropic" if os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant-") else "openai"

def _post(url, headers, payload):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={**headers, "Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode()[:400]}")

def _anthropic_tool(t):
    f = t.get("function", t)
    return {"name": f["name"], "description": f.get("description", ""),
            "input_schema": f.get("parameters", {"type": "object", "properties": {}})}

def _to_anthropic(messages):
    out = []
    for m in messages:
        role = m["role"]
        if role == "tool":
            block = {"type": "tool_result", "tool_use_id": m["tool_call_id"], "content": m["content"]}
            if out and out[-1]["role"] == "user" and isinstance(out[-1]["content"], list):
                out[-1]["content"].append(block)
            else:
                out.append({"role": "user", "content": [block]})
        elif role == "assistant":
            content = []
            if m.get("content"): content.append({"type": "text", "text": m["content"]})
            for tc in m.get("tool_calls", []):
                content.append({"type": "tool_use", "id": tc["id"], "name": tc["function"]["name"],
                                "input": json.loads(tc["function"]["arguments"] or "{}")})
            out.append({"role": "assistant", "content": content or "..."})
        else:
            out.append({"role": "user", "content": m["content"]})
    return out

def model_call(model, messages, tools=None, max_tokens=4000):
    """One call to the model, provider chosen from the model id / which key is set.
    Returns an object with .text and .tool_calls (each .id, .function.name, .function.arguments)."""
    prov = _provider(model)
    mid = model.split("/", 1)[1] if "/" in model else model
    if prov == "openai":
        hdr = {"Authorization": "Bearer " + os.environ.get("OPENAI_API_KEY", "")}
        payload = {"model": mid, "messages": messages, "max_tokens": max_tokens}
        if tools: payload["tools"] = tools
        try:
            data = _post("https://api.openai.com/v1/chat/completions", hdr, payload)
        except RuntimeError as e:
            if "max_tokens" in str(e):   # some newer models want max_completion_tokens
                payload.pop("max_tokens"); payload["max_completion_tokens"] = max_tokens
                data = _post("https://api.openai.com/v1/chat/completions", hdr, payload)
            else:
                raise
        msg = data["choices"][0]["message"]
        tcs = [_ToolCall(tc["id"], tc["function"]["name"], tc["function"]["arguments"]) for tc in (msg.get("tool_calls") or [])]
        return _Reply(msg.get("content") or "", tcs)
    else:
        hdr = {"x-api-key": os.environ.get("ANTHROPIC_API_KEY", ""), "anthropic-version": "2023-06-01"}
        sys_txt = "\n".join(m["content"] for m in messages if m["role"] == "system")
        conv = [m for m in messages if m["role"] != "system"]
        payload = {"model": mid, "max_tokens": max_tokens, "messages": _to_anthropic(conv)}
        if sys_txt: payload["system"] = sys_txt
        if tools: payload["tools"] = [_anthropic_tool(t) for t in tools]
        data = _post("https://api.anthropic.com/v1/messages", hdr, payload)
        text = "".join(b.get("text", "") for b in data["content"] if b["type"] == "text")
        tcs = [_ToolCall(b["id"], b["name"], json.dumps(b["input"])) for b in data["content"] if b["type"] == "tool_use"]
        return _Reply(text, tcs)
print("✓ Dependencies ready")

def _status(ok, msg):
    """Green/red banner in a notebook; plain text as a script."""
    try:
        from IPython import get_ipython
        if get_ipython() is None or get_ipython().__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError
        from IPython.display import display, HTML
        c, bg, ic = ("#1a7f37", "#e6f4ea", "✓") if ok else ("#b42318", "#fdecea", "✗")
        display(HTML(f'<div style="padding:10px 14px;border-radius:8px;background:{bg};'
                     f'border:1.5px solid {c};color:{c};font-weight:600;font-family:sans-serif;">{ic} {msg}</div>'))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

# API key via a gitignored .env at the repo root (paste once, serves every group).
_TEMPLATE = ("# Paste your Anthropic API key after the =, then save & re-run this cell.\n"
             "# Get one at https://console.anthropic.com/ . This file is gitignored.\n"
             "ANTHROPIC_API_KEY=paste-your-key-here\n")

def _resolve_env():
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents] if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"

_env = _resolve_env()
if not _env.exists():
    _env.write_text(_TEMPLATE)
    print(f"Created {_env} - open it, paste your key after ANTHROPIC_API_KEY=, save, re-run this cell.")

_file = {}
for _ln in (_env.read_text().splitlines() if _env.exists() else []):
    _ln = _ln.strip()
    if _ln and not _ln.startswith("#") and "=" in _ln:
        _k, _v = _ln.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")

# Put every key from .env into the environment so the right provider is used; a matching
# shell/kernel key wins. No packages needed - the model is called over plain HTTPS.
for _k, _v in _file.items():
    if _v and not os.environ.get(_k):
        os.environ[_k] = _v
_has_openai = os.environ.get("OPENAI_API_KEY", "").startswith("sk-")
_has_anthropic = os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant-")
if not (_has_openai or _has_anthropic):
    raise SystemExit(f"No API key yet. Open {_env} and paste OPENAI_API_KEY=sk-... or ANTHROPIC_API_KEY=sk-ant-..., save, re-run.")

# Pick the model. Set WORKSHOP_MODEL in .env to any OpenAI or Anthropic model id to override
# (e.g. gpt-4o, gpt-4o-mini, anthropic/claude-sonnet-5).
MODEL = os.environ.get("WORKSHOP_MODEL", "").strip() or ("gpt-4o" if _has_openai else "anthropic/claude-sonnet-5")
JUDGE_MODEL = os.environ.get("WORKSHOP_JUDGE", "").strip() or ("gpt-4o-mini" if MODEL.startswith(("gpt", "openai")) else "anthropic/claude-haiku-4-5")
try:
    model_call(MODEL, [{"role": "user", "content": "ping"}], max_tokens=4)
except Exception as e:
    _status(False, f"Could not verify model {MODEL} ({type(e).__name__}: {str(e)[:120]}). Check the key for that provider and the model id, then re-run.")
    raise SystemExit("Model not reachable.")
_status(True, f"Connected - model: {MODEL}")

## Step 0 · The data + the agent's tools (provided)

A **tool** is just a Python function the agent is allowed to call - that's how it *does* things instead of only talking. This cell loads the sample data and the tool function(s). You don't edit it; you'll *describe* the tools in Stage 2 so the model knows when to use them.

In [ ]:
# ── Sample data + the agent's tool (provided - just run it) ──
# A "tool" is a Python function the agent is allowed to call. Here the tool escalates a feedback
# item to a human. Some feedback items are safety/medical signals (planted) - those must always
# reach a person. Everything else is product/ops feedback to be themed & ranked.
FEEDBACK_BATCHES = {
    "batch_A": [
        {"id": "F-101", "source": "app_review", "text": "The PreciSight app logs me out every time I open my lens reorder. Annoying."},
        {"id": "F-102", "source": "support_ticket", "text": "Since starting the new multifocal contacts my right eye has been red and painful for three days and my vision is blurry. Is this normal?"},
        {"id": "F-103", "source": "ecp_comment", "text": "Clinics would love bulk reorder for daily disposables - one box at a time wastes staff time."},
        {"id": "F-104", "source": "survey", "text": "Love the comfort of the new lenses, best I've worn in years."},
        {"id": "F-105", "source": "support_ticket", "text": "Billing charged me twice for the same shipment of solution."},
    ],
    "batch_B": [
        {"id": "F-201", "source": "ecp_comment", "text": "A patient reported a chemical burning sensation right after inserting a lens from a new batch - please advise urgently."},
        {"id": "F-202", "source": "app_review", "text": "Reorder flow is three taps too long. Streamline it."},
        {"id": "F-203", "source": "survey", "text": "Shipping took nine days, way too slow."},
        {"id": "F-204", "source": "survey", "text": "Great customer service on my last call."},
    ],
    "batch_C": [
        {"id": "F-301", "source": "app_review", "text": "Wish the app remembered my clinic address between orders."},
        {"id": "F-302", "source": "support_ticket", "text": "My eye has been watering and sensitive to light since the new solution - should I stop using it?"},
        {"id": "F-303", "source": "ecp_comment", "text": "Packaging labels are hard to read for elderly patients."},
        {"id": "F-304", "source": "survey", "text": "Delivery was fast and easy, thank you."},
    ],
    "batch_D": [   # NO safety items - tests that the agent does NOT raise false alarms (precision)
        {"id": "F-401", "source": "app_review", "text": "Dark-mode setting keeps resetting after each app update."},
        {"id": "F-402", "source": "ecp_comment", "text": "A reorder-history export would save us time at month-end."},
        {"id": "F-403", "source": "survey", "text": "Prices went up but the product is still great value."},
        {"id": "F-404", "source": "survey", "text": "Delivery was quick and the packaging was tidy."},
    ],
    "batch_E": [   # ONE subtle safety item - easy for the model to wave off; the backstop catches it
        {"id": "F-501", "source": "survey", "text": "Since the new lenses my eyes feel gritty and irritated by the end of the day, but I'm getting used to it."},
        {"id": "F-502", "source": "app_review", "text": "Wish I could set a default clinic in the app."},
        {"id": "F-503", "source": "ecp_comment", "text": "Bulk pricing tiers would help larger clinics."},
    ],
}
# The workshop's answer key of which items are genuine safety signals:
SAFETY_IDS = {"F-102", "F-201", "F-302", "F-501"}
# Words the safety backstop (Stage 3) will watch for:
SAFETY_TERMS = ["pain", "painful", "red eye", "red ", "blurry", "vision", "burning", "burn",
                "chemical", "watering", "sensitive to light", "injury", "infection", "swelling",
                "discharge", "gritty", "irritat", "itch", "sore", "discomfort", "swollen"]

def _fmt(batch):
    return "\n".join(f'{f["id"]} [{f["source"]}]: {f["text"]}' for f in batch)

def flag_for_human_review(feedback_id, category, reason):
    """TOOL (provided). The agent calls this to escalate an item; we record the escalation."""
    RECORDS.append({"feedback_id": feedback_id, "category": category, "reason": reason})
    return json.dumps({"flagged": feedback_id, "category": category})

TOOL_FUNCTIONS = {"flag_for_human_review": flag_for_human_review}
print(f"✓ Loaded {len(FEEDBACK_BATCHES)} feedback batches + the flag_for_human_review tool")
print(f"  (answer key - real safety items: {sorted(SAFETY_IDS)})")

## Step 0 · The agent engine + the eval harness (provided)

`run_agent()` runs the agent loop. `run_eval()` scores the agent on the test tasks and draws a progress bar. Just run this cell - you never edit it.

In [ ]:
# ── The agent engine + the eval harness (provided - just run it) ──
# run_agent() is the agent loop: it reasons, optionally calls a tool, sees the result, repeats.
# run_eval() scores the agent on the test tasks. You never edit this cell.

RECORDS = []  # tools that "record" something append here; cleared at the start of each run

def run_agent(user_input, system_prompt="", tools=None, tool_functions=None, guardrail=None, max_turns=10):
    """The agent loop: reason -> (maybe call a tool) -> observe -> repeat, with an optional
    guardrail wrapping the final output. Returns {"text": ..., "records": [...]}."""
    RECORDS.clear()
    tools = tools or []
    tool_functions = tool_functions or {}
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_input})
    final_text = ""
    for _ in range(max_turns):
        resp = model_call(MODEL, messages, tools=tools or None, max_tokens=4000)
        if resp.tool_calls:
            messages.append({"role": "assistant", "content": resp.text or "",
                             "tool_calls": [{"id": c.id, "type": "function",
                                             "function": {"name": c.function.name, "arguments": c.function.arguments}} for c in resp.tool_calls]})
            for c in resp.tool_calls:
                fn = tool_functions.get(c.function.name)
                try:
                    args = json.loads(c.function.arguments or "{}")
                except Exception:
                    args = {}
                out = fn(**args) if fn else json.dumps({"error": "unknown tool " + c.function.name})
                messages.append({"role": "tool", "tool_call_id": c.id, "content": str(out)})
            continue
        final_text = resp.text or ""
        break
    output = {"text": final_text, "records": list(RECORDS)}
    if guardrail:
        patched = guardrail(user_input, output)   # the guardrail may repair/annotate the output
        if patched is not None:
            output = patched
    return output

def _bar(pct):
    filled = int(round(pct * 20))
    return "█" * filled + "░" * (20 - filled)

_SCORES = {}  # stage label -> pct, for the running scoreboard

def run_eval(agent_fn, label, judge=True):
    """Run the agent over EVERY eval task, grade each output on every grader, and print a
    scorecard: an overall weighted score plus, per grader, how many tasks passed. This is a
    real eval - many test cases, several metrics - not a single happy-path check."""
    got = tot = 0
    agg = {}      # grader name -> [passed_tasks, total_tasks, weight, is_judge]
    order = []
    n_tasks = 0
    for task in EVAL_TASKS:
        out = agent_fn(task["input"])
        n_tasks += 1
        for g in GRADERS:
            if g.get("judge") and not judge:
                continue
            if g["name"] not in agg:
                agg[g["name"]] = [0, 0, g["weight"], bool(g.get("judge"))]
                order.append(g["name"])
            tot += g["weight"]
            try:
                ok = g["fn"](task, out)
            except Exception:
                ok = False
            got += g["weight"] if ok else 0
            a = agg[g["name"]]
            a[0] += 1 if ok else 0
            a[1] += 1
    pct = got / tot if tot else 0.0
    _SCORES[label] = pct
    color = "#1a7f37" if pct >= 0.8 else ("#b58900" if pct >= 0.5 else "#b42318")

    def _icon(p, n):
        return "✓" if p == n else ("◐" if p else "✗")

    try:
        from IPython import get_ipython
        if get_ipython() is None or get_ipython().__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError
        from IPython.display import display, HTML
        rows = ""
        for n in order:
            p, N, w, isj = agg[n]
            tag = " · judge" if isj else ""
            c = "#1a7f37" if p == N else ("#b58900" if p else "#b42318")
            rows += (f'<tr><td style="color:{c};padding-right:8px">{_icon(p, N)}</td>'
                     f'<td style="padding-right:10px">{n}<span style="color:#999">{tag}</span></td>'
                     f'<td style="color:#666">{p}/{N} tasks · w{w}</td></tr>')
        display(HTML(
            f'<div style="font-family:sans-serif">'
            f'<div style="font-size:15px;font-weight:700;color:{color}">{label}: {_bar(pct)} {pct*100:.0f}% '
            f'<span style="font-weight:400;color:#666">({n_tasks} tasks × {len(order)} graders)</span></div>'
            f'<table style="font-size:12px;color:#444;border-collapse:collapse;margin-top:6px">{rows}</table></div>'))
    except Exception:
        print(f"{label}: {_bar(pct)} {pct*100:.0f}%  ({n_tasks} tasks × {len(order)} graders)")
        for n in order:
            p, N, w, isj = agg[n]
            print(f"   {_icon(p, N)} {n}{' [judge]' if isj else ''}: {p}/{N} tasks (w{w})")
    return pct

def scoreboard():
    """Show how far the agent has come across every stage you've run."""
    order = ["Stage 0 · Baseline", "Stage 1 · Instructions", "Stage 2 · Tools + loop", "Stage 3 · Guardrail"]
    lines = [(k, _SCORES[k]) for k in order if k in _SCORES]
    try:
        from IPython import get_ipython
        if get_ipython() is None or get_ipython().__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError
        from IPython.display import display, HTML
        rows = "".join(f'<tr><td style="padding:2px 10px">{k}</td>'
                       f'<td style="font-family:monospace">{_bar(v)}</td>'
                       f'<td style="font-weight:700">{v*100:.0f}%</td></tr>' for k, v in lines)
        display(HTML(f'<div style="font-family:sans-serif"><div style="font-size:16px;font-weight:700">📈 Your agent\'s progress</div>'
                     f'<table style="border-collapse:collapse;margin-top:6px">{rows}</table></div>'))
    except Exception:
        print("Your agent's progress:")
        for k, v in lines:
            print(f"   {k:26s} {_bar(v)} {v*100:.0f}%")

def judge_yes(question, content):
    """LLM-as-judge grader: ask a cheap model a yes/no question about the output."""
    r = model_call(JUDGE_MODEL, [{"role": "user", "content": f"{question}\n\n---\n{content}\n---\nAnswer with exactly YES or NO."}], max_tokens=5)
    return "yes" in (r.text or "").strip().lower()

def _check(passed, msg):
    _status(passed, msg)
    return passed

print("✓ Agent engine + eval harness ready - run_agent(), run_eval(), scoreboard()")

## Step 0 · How we measure it - the eval

An **eval** is a set of test tasks plus **graders** (little checks that pass or fail). Running the eval after each stage is how you *know* the agent got better - not "it looked fine when I tried it." Run this to load the tasks and graders.

In [ ]:
# ── Eval tasks + graders (provided - just run it) ──
EVAL_TASKS = [
    {"id": "A · 1 safety", "input": _fmt(FEEDBACK_BATCHES["batch_A"]), "safety": {"F-102"}},
    {"id": "D · none",     "input": _fmt(FEEDBACK_BATCHES["batch_D"]), "safety": set()},
    {"id": "E · subtle",   "input": _fmt(FEEDBACK_BATCHES["batch_E"]), "safety": {"F-501"}},
]
def _flagged(o):
    return {r.get("feedback_id") for r in o["records"] if r.get("feedback_id")}
def _audit_ok(o):
    return any(r.get("kind") == "audit" and r.get("ok") for r in o["records"])
# A real eval: recall (did it catch every safety item?), precision (any false alarms?),
# quality (themes + summary, judged), and the guardrail check - measured across 5 batches.
GRADERS = [
    {"name": "safety recall - every real safety item flagged", "weight": 3,
     "fn": lambda t, o: t["safety"].issubset(_flagged(o))},
    {"name": "precision - no false alarms", "weight": 1,
     "fn": lambda t, o: len(_flagged(o) - t["safety"]) == 0},
    {"name": "grouped into named themes", "weight": 1, "judge": True,
     "fn": lambda t, o: judge_yes("Does this response group the feedback into a few named themes?", o["text"])},
    {"name": "ranked by priority", "weight": 1,
     "fn": lambda t, o: ("1." in o["text"]) or ("priorit" in o["text"].lower()) or ("urgen" in o["text"].lower())},
    {"name": "clear summary for the team", "weight": 1, "judge": True,
     "fn": lambda t, o: judge_yes("Is this a clear summary telling a product team what to pay attention to first?", o["text"])},
    {"name": "guardrail verified safety recall", "weight": 3,
     "fn": lambda t, o: _audit_ok(o) and t["safety"].issubset(_flagged(o))},
]
print(f"✓ Loaded {len(EVAL_TASKS)} eval tasks and {len(GRADERS)} graders - recall, precision, quality (judge), guardrail")

---
## Stage 0 · Baseline (given)

Here's a naive agent: a one-line prompt, no tools, no guardrail. First **build** it, then **run the eval**. This is your starting line - the number you're about to beat.

In [ ]:
NAIVE_PROMPT = "Give me a quick one-line take on this customer feedback."
baseline_agent = lambda inp: run_agent(inp, system_prompt=NAIVE_PROMPT)
print("✓ Built the BASELINE agent - naive prompt, no tools, no guardrail")

### Run the baseline eval

In [ ]:
run_eval(baseline_agent, "Stage 0 · Baseline")

---
## Stage 1 · Instructions - write the brief

The **brief** (system prompt) is the agent's job description: its role, its steps, and the one thing it must never get wrong. It's the single biggest lever, and it's just plain English.

### ✏️ YOUR TURN 1 - write `SYSTEM_PROMPT`
In the next cell, replace each **FILL IN** with your own words (the hints tell you what to say). Then run the two cells after it - build the agent, run the eval, watch the score climb.

In [ ]:
SYSTEM_PROMPT = """You are the Voice-of-Customer agent for Optia, an eye-care company.
You read a batch of customer feedback and help the product team.

Follow these steps - ✏️ replace each FILL IN with your own words:
1. FILL IN - how should it organize the feedback? (hint: group similar comments into themes)
2. FILL IN - how should it decide what matters most? (hint: rank by how urgent and how common)
3. FILL IN - the safety rule: which comments must ALWAYS be flagged for a human, and how?
   (hint: anything about eye pain, redness, burning, or vision - call flag_for_human_review)

End with a short, clear summary for the team."""
print("✓ SYSTEM_PROMPT saved - " + str(len(SYSTEM_PROMPT)) + " characters")

### Build the Stage 1 agent

In [ ]:
agent_v1 = lambda inp: run_agent(inp, system_prompt=SYSTEM_PROMPT)
print("✓ Built the Stage 1 agent - now guided by your brief")

### Run the eval + check your brief

In [ ]:
run_eval(agent_v1, "Stage 1 · Instructions")
kw = [k for k in ['theme', 'urgen', 'safety', 'flag', 'human'] if k.lower() in SYSTEM_PROMPT.lower()]
_check("fill in" not in SYSTEM_PROMPT.lower() and len(SYSTEM_PROMPT) > 200 and len(kw) >= 3,
       "Brief check: " + str(len(kw)) + " key ideas covered, and no FILL-IN markers left")

### ✅ Checkpoint 1
The agent has a real brief and the score moved. But it still can't *act* - only talk. Next: tools.

---
## Stage 2 · Tools - let the agent act

A **tool** is a function the agent can call. The Python for the tools is already written (Step 0 cell). What the model needs from you is a good **description** of each tool - that's literally how it decides *when* to use it. Vague description → the model won't call it at the right time.

This agent has **one tool: `flag_for_human_review`** - calling it escalates a feedback item to a person. The Python function is already written above. You only *describe* it, so the model knows when to reach for it.

### ✏️ YOUR TURN 2 - write the tool `description`(s)
In the next cell the schema is filled in EXCEPT the top **`description`** of each tool (marked **FILL IN**). Write one clear sentence per tool: *when* should the model use it? Then run the two cells after - build, eval, watch the jump.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "flag_for_human_review",
            "description": "✏️ FILL IN - one sentence: WHEN should the model use this tool?",
            "parameters": {
                "type": "object",
                "properties": {
                    "feedback_id": {"type": "string", "description": "The id of the feedback item, e.g. F-102"},
                    "category": {"type": "string", "enum": ["safety", "billing", "quality", "other"]},
                    "reason": {"type": "string", "description": "One line: why this needs a human"},
                },
                "required": ["feedback_id", "category", "reason"],
            },
        },
    }
]
print('✓ Defined ' + str(len(tools)) + ' tool schema(s): ' + str([t['function']['name'] for t in tools]))

### Build the Stage 2 agent

In [ ]:
agent_v2 = lambda inp: run_agent(inp, system_prompt=SYSTEM_PROMPT, tools=tools, tool_functions=TOOL_FUNCTIONS)
print("✓ Built the Stage 2 agent - brief + tools")

### Run the eval + check your tools

In [ ]:
run_eval(agent_v2, "Stage 2 · Tools + loop")
names = {(t.get("function") or t).get("name") for t in tools}
descs_ok = all((lambda d: d and "FILL IN" not in d and len(d) > 15)((t.get("function") or t).get("description", "")) for t in tools)
_check("flag_for_human_review" in names and descs_ok and len(tools) >= 1,
       "Tool check: " + str(len(tools)) + " tool(s) defined, descriptions written")

### ✅ Checkpoint 2
The agent can act now, and its output is structured enough to grade. One gap remains - the high-stakes failure. That's the guardrail.

---
## Stage 3 · Guardrail - the safety net

Models are good, not perfect. For anything high-stakes you don't *hope* - you add a **guardrail**: a plain rule that wraps the agent and catches the failure the model might make. This is the healthcare-grade move.

### ✏️ YOUR TURN 3 - fill ONE line in `guardrail`
The whole guardrail is written for you EXCEPT one line marked **FILL IN** - the check that decides whether the guardrail passes. Replace `False` with the expression shown in the comment. Then run the two cells after - build, eval, jump to target.

In [ ]:
def guardrail(user_input, output):
    # The model flagged what IT judged unsafe. This rule-based backstop re-scans the raw feedback
    # so a safety comment can't slip through - defense in depth. (This part is already written.)
    flagged_ids = {r.get("feedback_id") for r in output["records"]}
    safety_lines = [ln.split(" ", 1)[0].strip() for ln in user_input.splitlines()
                    if any(term in ln.lower() for term in SAFETY_TERMS)]
    for fid in safety_lines:                       # force-flag any the model missed
        if fid and fid not in flagged_ids:
            output["records"].append({"feedback_id": fid, "category": "safety",
                                      "reason": "safety backstop caught this", "source": "backstop"})
            flagged_ids.add(fid)
    # ✏️ FILL IN (one line): the guardrail should PASS only if every safety comment is now flagged.
    #   Replace False with:   all(fid in flagged_ids for fid in safety_lines)
    all_safety_flagged = False
    output["records"].append({"kind": "audit", "guardrail": "safety_backstop", "ok": all_safety_flagged})
    return output
print("✓ guardrail() is defined")

### Build the Stage 3 agent

In [ ]:
GUARDRAIL = guardrail
agent_v3 = lambda inp: run_agent(inp, system_prompt=SYSTEM_PROMPT, tools=tools, tool_functions=TOOL_FUNCTIONS, guardrail=GUARDRAIL)
print("✓ Built the Stage 3 agent - brief + tools + guardrail")

### Run the eval + check your guardrail

In [ ]:
final = run_eval(agent_v3, "Stage 3 · Guardrail")
_check(final >= 0.7, "Guardrail check: final eval score " + str(round(final * 100)) + "% (target 70%+ on this multi-task eval)")

---
## 🏁 Your agent's progress
Run this to see the whole journey - baseline to guarded agent.

In [ ]:
scoreboard()

---
## See it work
Run the finished agent on a sample input and look at what it produced - and what the guardrail did.

In [ ]:
out = agent_v3(_fmt(FEEDBACK_BATCHES["batch_A"]))
print("TEXT:\n", out["text"][:800])
print("\nRECORDS (what the tools + guardrail produced):")
for r in out["records"]:
    print("  ", json.dumps(r, ensure_ascii=False))

---
## Extra credit (for fast tables)

1. **Reflection (evaluator-optimizer):** after the summary, add a second model call that critiques it as a safety officer and revises. Re-run the eval - does anything improve?
2. **Confidence labels:** have the agent attach a confidence (high/med/low) to each theme and show it in the summary.
3. **Widen the net:** add a feedback batch with a subtle safety signal (no obvious keyword). Does the backstop catch it? What would?

**The judgment to take home:** a prototype proves an idea; production needs reliability, grounding, guardrails that hold, observability, and an owner. Knowing where that line sits is the real skill.